# S6E10 | Push Model + Orthogonal Directions | LB 0.96223

This notebook rebuilds my public **0.96223** file from public inputs only. It has two parts.

1. **Push model → 0.96217.** Start from goodpjw2008's public 0.96213 file `P` and push it *away* from ten weaker files whose public scores are known.
2. **Orthogonal directions → 0.96223.** Add two public improvements that point in directions orthogonal to the push. They come from MAYANK RAJ's Mirror Pair Climb (0.96218) and goodpjw2008's climb file (0.96217).

**Please read the caveat at the end before you pick this as a final submission.**

Credits:
- [goodpjw2008](https://www.kaggle.com/code/goodpjw2008/s6e10-meta-stack-on-public-oof-lb-0-96217): the 0.96213 file, the probe files, the 0.96217 climb file, and the push model idea.
- [MAYANK RAJ](https://www.kaggle.com/code/mayankraaz/s6e10-mirror-pair-climb-0-96218): the Mirror Pair Climb file.

In [ ]:
import glob, os
import numpy as np, pandas as pd
from scipy.stats import rankdata

ROOTS = os.environ.get("PUSH_ROOTS", "/kaggle/input").split(os.pathsep)

def find(name, hint=""):
    hits = [h for r in ROOTS for h in glob.glob(os.path.join(r, "**", name), recursive=True) if hint in h.replace("\\", "/")]
    assert hits, f"input not found: {name} {hint}"
    return sorted(hits)[0]

te = pd.read_csv(find("test.csv", "playground-series-s6e10"), usecols=["id"])["id"].values
pct = lambda v: (rankdata(v) - 0.5) / len(v)                      # every file lives on the percentile-rank scale
L = lambda f, hint="": pct(pd.read_csv(find(f, hint)).set_index("id").iloc[:, 0].loc[te].values)

## 1. Push model

Every file is a percentile rank. For a weaker file `X_k` with public score `S_k`, the candidate is

`B = P + Σ ε_k (P − X_k)`

A second-order model of the public AUC gives

`gain(ε) = g·ε − K·c·εᵀΣε`

with these terms:
- `Σ` is the covariance of the differences `X_k − P`.
- `g_k = S_P − S_k − 4·c_k·Σ_kk`.
- `c_k` is how much the file would add in a 50/50 blend.

The best push is `ε = Σ⁻¹g / (2Kc)`.

I fitted `K` on 12 public probes; `K = 10` fits them with an RMSE of 0.8e-5. The per-file `c_k` values come from two sources. For goodpjw's files, I used the values in his notebook. For the three singles that I measured myself, I estimated them on OOF.

In [ ]:
P = L("meta_part4_096213_1023.csv"); SP = 0.96213
COMP = {  # name: (file, public score, c_k)
    "xgb_base": ("probe_comp_xgb_base_raw.csv",      0.95808, .0909),
    "gen12":    ("probe_comp_gen_v2_o12.csv",        0.95343, .1021),
    "realmlp":  ("probe_comp_realmlp_v4_s1.csv",     0.96082, .0858),
    "gen3":     ("probe_comp_gen3_pilot.csv",        0.95662, .0927),
    "tp35":     ("m_tabpfn35.csv",                   0.96064, .088),
    "ku":       ("m_kumolarge.csv",                  0.95956, .080),
    "te50":     ("m_xgb_te2f50.csv",                 0.96067, .087),
    "p2viewb":  ("probe_comp_revisit_p2_viewb.csv",  0.96129, .085),
    "ens_v4":   ("probe_comp_ens_v4_k10.csv",        0.96107, .085),
    "ne4f":     ("probe_comp_ne4f_single.csv",       0.96110, .088),
}
keys = list(COMP); X = {k: L(f) for k, (f, _, _) in COMP.items()}
D = np.column_stack([X[k] - P for k in keys]); Dc = D - D.mean(0)
Sig = Dc.T @ Dc / len(P)
g = np.array([SP - COMP[k][1] - 4 * COMP[k][2] * Sig[i, i] for i, k in enumerate(keys)])

K, c = 10, 0.09
eps = np.linalg.solve(Sig, g) / (2 * K * c)
print("predicted public:", round(SP + g @ eps - K * c * eps @ Sig @ eps, 6))
print(pd.Series(eps * 100, keys).round(1).to_string())
R1 = pct(P + sum(e * (P - X[k]) for k, e in zip(keys, eps)))

`R1` scored **0.96217** on the public LB; the model predicted 0.962169. A smaller version that uses only the base four plus `tp35`, `ku` and `te50` also scored 0.96217.

## 2. Orthogonal directions

- `M − P`, from Mirror Pair Climb (0.96218): its cosine with `R1 − P` is −0.01.
- `Q − P`, from goodpjw's climb file (0.96217): about 0.65·(R1 − P) plus a residual `Qp` that is orthogonal to both `R1 − P` and `M − P`.

Because the directions are orthogonal, their gains roughly add. Measured on the public LB:

| file | public |
|---|---|
| R1 + (M − P) | 0.96222 |
| R1 + 1.3(M − P) | 0.96222 |
| R1 + (M − P) + Qp | 0.96222 |
| **R1 + 1.3(M − P) + Qp** | **0.96223** |
| R1 + 1.6(M − P) + 1.5Qp | 0.96222 |
| R1 + 1.3(M − P) + 1.6Qp | 0.96222 |

In [ ]:
M = L("submission.csv", "mirror-pair-climb")
Q = L("meta_part6_climb1_096217_1030.csv")
A = np.column_stack([R1 - P, M - P])
coef = np.linalg.lstsq(A, Q - P, rcond=None)[0]; Qp = (Q - P) - A @ coef
print("Q - P on [R1 - P, M - P]:", coef.round(3))

T3 = pct(R1 + 1.3 * (M - P) + Qp)
for name, F in [("R1", R1), ("T3", T3)]:
    ref = L({"R1": "ref_r1_push_096217.csv", "T3": "ref_t3_096223.csv"}[name])
    print(name, "max |rank diff| vs my scored file:", float(np.abs(F - ref).max()))
pd.DataFrame({"id": te, "satisfaction": T3}).to_csv("submission.csv", index=False)

## Caveat for the final two

This is a public-LB file. Its gains are not the same kind:

- **Push (0.96213 → 0.96217).** This moves away from weaker models along directions the model supports. I expect it to carry over to private roughly unchanged. Its rank difference from `P` is small, about 0.006.
- **Mirror direction (+0.5e-4).** This includes block noise whose signs were chosen on the public LB. On private, that part is noise: at best it does nothing, and it may cost a little.

For a safer final, `R1` (0.96217) keeps the push without the public-selected noise.